# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aisyahnabillah/ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

**Finding 1:** "What Predicts Health?" (ML Appendix — Feature Importance)

The paper reports **Random Forest feature importance for predicting health score**, with **Average Position at 43%** and **Impressions at 32%**, combining for 75% of the model's decision weight. To its credit, the paper's own caption already discloses a concern: "the target itself is partly constructed from some of these inputs, so importance is descriptive rather than causal."

**My methodology question:** since the label is built partly FROM position and impressions, how much of that 75% importance is a **real learned pattern** versus the model reading a piece of the label back off itself? The standard test for this is retraining **without** the suspect features to see if the score collapses, that comparison isn't shown here.

**Finding 2**: "What Predicts Growth?" (ML Appendix — Growth & Classification)

The paper reports **a logistic regression reaching 71% holdout accuracy predicting growing vs. declining pages, with Content Age as the strongest negative signal.**

**My methodology question**: was the holdout split **grouped by brand**, or a plain random row-level split? If brands appear in both train and holdout, the model may partly be learning **brand-level quirks**, not a generalizable signal. I'd also ask what the **base rate** was in the holdout (what share of pages were actually growing), since 71% accuracy on a label that's already, say, 60% one class reflects far less real skill than 71% against a 50/50 split. The paper doesn't state either detail, so the 71% figure is hard to fully evaluate as-is, though the direction of the finding (content age as a negative signal) is plausible and consistent with Finding #2 in the main paper (the age/decay lifecycle pattern).

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

features = ["content_age_days", "days_since_last_update", "impressions_90d",
            "avg_position", "ctr", "word_count", "search_volume", "engagement_rate"]
X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)
y = df["is_declining_label"]

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

# BEFORE: plain random split (no grouping) — the naive way most people split
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X, y, test_size=0.3, random_state=42)
rf_random = RandomForestClassifier(n_estimators=200, max_depth=6, class_weight="balanced", random_state=42)
rf_random.fit(X_train_r, y_train_r)
scores_random = rf_random.predict_proba(X_test_r)[:, 1]
p20_random = precision_at_k(scores_random, y_test_r.values, 20)

# AFTER: client-grouped split — the same one used in ML-08
groups = df["client_id"]
gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))
X_train_g, X_test_g = X.iloc[train_idx], X.iloc[test_idx]
y_train_g, y_test_g = y.iloc[train_idx], y.iloc[test_idx]
rf_grouped = RandomForestClassifier(n_estimators=200, max_depth=6, class_weight="balanced", random_state=42)
rf_grouped.fit(X_train_g, y_train_g)
scores_grouped = rf_grouped.predict_proba(X_test_g)[:, 1]
p20_grouped = precision_at_k(scores_grouped, y_test_g.values, 20)

print(f"BEFORE (random split)  precision@20: {p20_random:.3f}")
print(f"AFTER  (client-grouped) precision@20: {p20_grouped:.3f}")
print(f"Gap: {p20_random - p20_grouped:.3f}")

BEFORE (random split)  precision@20: 0.950
AFTER  (client-grouped) precision@20: 0.450
Gap: 0.500


The gap is large, and the direction confirms memorization. With a plain random split, pages from the same client can land in both train and test, so the model partly learns client-specific quirks instead of a pattern that generalizes to a client it has never seen. The 0.950 score looked almost perfect, which itself should have been suspicious rather than celebrated. Once tested on truly unseen clients (grouped split), the honest number drops to 0.450, this is the number that actually reflects what the model can do in a real deployment, where every client's content is new to the model at some point.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Attack checklist, run against my final Week-5/8 feature set
print("Features used:", features)
print("trend_direction/trend_pct in features?", 
      any(f in ["trend_direction", "trend_pct"] for f in features))
print("Any product-decision flags (health_score, priority_score, action_type) in dataset?",
      any(c in df.columns for c in ["health_score", "priority_score", "action_type"]))

# Collapse test: retrain WITH a deliberately leaky feature to confirm the test harness itself works
# Perbaikan: leak yang BENAR untuk is_declining_label harus dari trend_pct, bukan ctr
df["trend_pct_leak"] = df["trend_pct"]
X_leaky2 = df[features + ["trend_pct_leak"]].fillna(0)

X_train_l2, X_test_l2, y_train_l2, y_test_l2 = train_test_split(X_leaky2, y, test_size=0.3, random_state=42)
rf_leak2 = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_train_l2, y_train_l2)
leak_score2 = rf_leak2.predict_proba(X_test_l2)[:, 1]
print(f"Sanity check — precision@20 WITH real leak (trend_pct): {precision_at_k(leak_score2, y_test_l2.values, 20):.3f}")

Features used: ['content_age_days', 'days_since_last_update', 'impressions_90d', 'avg_position', 'ctr', 'word_count', 'search_volume', 'engagement_rate']
trend_direction/trend_pct in features? False
Any product-decision flags (health_score, priority_score, action_type) in dataset? False
Sanity check — precision@20 WITH real leak (trend_pct): 1.000


In [6]:
print(f"Base rate (random test): {y_test_r.mean():.3f}")
print(f"Base rate (grouped test): {y_test_g.mean():.3f}")

Base rate (random test): 0.545
Base rate (grouped test): 0.559


Leakage audit checklist:
- [✅] trend_direction / trend_pct excluded from the honest feature set (label-derived, per ML-04)
- [✅] No product-decision flags exist in this dataset
- [✅] All 8 features are knowable before the decision moment
- [✅] Split is grouped by client_id, not random, confirmed by the Section 2 before/after test
- [✅] Base rate printed: 0.545 (random test) and 0.559 (grouped test), close to each other, so the honest precision@20 of 0.450 is actually slightly BELOW the base rate, an important honest finding in itself (see note below)
- [✅] Test harness confirmed working: my first attempt used ctr_leak, which did NOT push the score toward 1.0 (it landed at 0.850, close to the honest baseline), a useful reminder that a column only counts as "a leak" if it's actually derived from THIS specific label. ctr is not what is_declining_label was built from, trend_pct is. Retrying with trend_pct_leak (the real leak source) pushed precision@20 to 1.000, confirming the test harness genuinely catches leakage when it's actually present.

Additional honest note: the grouped-split precision@20 (0.450) is close to the base rate (0.559), meaning on unseen clients, this feature set is only marginally better than picking pages at random for the top 20. This is a more honest and more useful finding than a clean leakage checklist alone, it tells me the current feature set needs real improvement, not just validation hygiene.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

My boldest claim so far (from ML-08): "The baseline rule from Week 4 outperformed both models 
at precision@20."

Rewritten in safe language: "In this observed comparison, on this dataset and this client-grouped split, the Week-4 rule-based baseline showed a higher precision@20 (0.65) than either the Decision Tree (0.60) or Random Forest (0.45) I trained in Week 5. This is a directional, decision-support finding specific to this data slice and this split, not a general claim that simple rules always beat learned models."

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.